<a href="https://colab.research.google.com/github/patelomniraj/ML-Projects/blob/main/From_Encoding_Distortion_to_Production_Grade_ML_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset: Adult Income Dataset (UCI Machine Learning Repository)

Link: https://archive.ics.uci.edu/dataset/2/adult


---


# Problem Statement:

For each measurement type, list which statistical operations are valid (e.g., you cannot take a mean of nominal data) and which encoding strategy is production-correct (label encoding vs one-hot vs ordinal encoding vs target encoding).
Implement all 4 encodings on real columns and show why using the wrong one distorts a downstream model.
---



# Libraries

In [127]:
from sklearn import set_config
set_config(display="diagram")

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler,TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Phase 1

In [128]:
# Import Data

# Raw Data URL from UCL Repository

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = ["age","workclass","fnlwgt","education","education-num","marital-status","occupation","relationship","race","sex","capital-gain","capital-loss","hours-per-week","native-country","income"]

  # Load Data
df = pd.read_csv(url,names=columns,skipinitialspace=True)

In [129]:
# Income Distribution

print("Income Distribution % :\n",df['income'].value_counts(normalize=True) * 100 )

Income Distribution % :
 income
<=50K    75.919044
>50K     24.080956
Name: proportion, dtype: float64


# Phase 2

In [130]:
# Income Convert Binary
df['income_binary'] = df['income'].apply(lambda x: 1 if str(x).strip().lower() == '>50k' else 0)

# 2. Select Feature
X = df[["age","relationship","hours-per-week"]]
y = df['income_binary']


# Train Test Split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3)


# **Distorted Version - Using Ordinal Encoder for Nominal Values**er

In [131]:

#  Apply different Encoders to different columns
distorted_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['age']),
        ('Hours', StandardScaler(), ['hours-per-week']),
        ('label_encoded_relationship',OrdinalEncoder(), ['relationship']),
    ])

distorted_model = Pipeline([
    ('preprocessor', distorted_preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=25))
])


# We are trying to give Relationship values - like  Child 0, Husband 1, Not in Family 2, Wife 3....

# Drawback - while using KNeighborsClassifier is useless. to provide data

distorted_model.fit(X_train,y_train)

y_pred = distorted_model.predict(X_test)

dis_accuracy = accuracy_score(y_test,y_pred)

print("Distorted Model Accuracy :", dis_accuracy)

Distorted Model Accuracy : 0.7796089671409561


In [132]:
# Corrected Way to Transform Nominal Data --> Apply OneHotEncoder , use dimentions for NeuralNetwork

correct_preprocessor = ColumnTransformer(
    transformers = [
    ('Age',StandardScaler(),['age']),
     ('Hours', StandardScaler(), ['hours-per-week']),
    ('OHE_Relationship',OneHotEncoder(),['relationship'])
])

correct_model = Pipeline([
    ('Preprocessor',correct_preprocessor),
    ('Model',KNeighborsClassifier(n_neighbors=25))
])

correct_model.fit(X_train,y_train)

y_pred = correct_model.predict(X_test)

correct_accuracy = accuracy_score(y_test,y_pred)

print("Correct Accuracy :",correct_accuracy)

Correct Accuracy : 0.7766403930801515


# Phase 3 Advance Encoding

In [133]:

# Select Feature
X = df[["relationship","education","native-country","capital-gain"]]
y = df['income_binary']


# Train Test Split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3)


In [134]:

# Defining Education from Lower To High Explicitly

education_levels = [
    'Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th',
    'HS-grad', 'Some-college', 'Assoc-voc', 'Assoc-acdm', 'Bachelors', 'Masters',
    'Prof-school', 'Doctorate'
]

In [135]:
# Master Preprocessor

master_processor = ColumnTransformer( [
    ('OHR_Relationship',OneHotEncoder(sparse_output=False),['relationship']),
    ('Capital Gain',StandardScaler(),['capital-gain']),
    ('Education_Encoded',OrdinalEncoder(categories=[education_levels]),['education']),
    ('Country-Encoded',TargetEncoder(),['native-country'])
],
    remainder='passthrough'
    )
#X_transformed = master_processor.fit_transform(X_train, y_train)
#print(X_transformed.shape)
#print("Preprocessing complete! Data is now perfectly encoded.")

master_processor

ColumnTransformer(remainder='passthrough',
                  transformers=[('OHR_Relationship',
                                 OneHotEncoder(sparse_output=False),
                                 ['relationship']),
                                ('Capital Gain', StandardScaler(),
                                 ['capital-gain']),
                                ('Education_Encoded',
                                 OrdinalEncoder(categories=[['Preschool',
                                                             '1st-4th',
                                                             '5th-6th',
                                                             '7th-8th', '9th',
                                                             '10th', '11th',
                                                             '12th', 'HS-grad',
                                                             'Some-college',
                                                             'Assoc-voc',
                                                             'Assoc-acdm',
                                                             'Bachelors',
                                                             'Masters',
                                                             'Prof-school',
                                                             'Doctorate']]),
                                 ['education']),
                                ('Country-Encoded', TargetEncoder(),
                                 ['native-country'])])

In [136]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('Preprocesssor',master_processor),
    ('Random Forest',RandomForestClassifier(n_estimators=300,random_state=1))])


pipe

Pipeline(steps=[('Preprocesssor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('OHR_Relationship',
                                                  OneHotEncoder(sparse_output=False),
                                                  ['relationship']),
                                                 ('Capital Gain',
                                                  StandardScaler(),
                                                  ['capital-gain']),
                                                 ('Education_Encoded',
                                                  OrdinalEncoder(categories=[['Preschool',
                                                                              '1st-4th',
                                                                              '5th-6th',
                                                                              '7th-8th',
                                                                              '9th',
                                                                              '10th',
                                                                              '11th',
                                                                              '12th',
                                                                              'HS-grad',
                                                                              'Some-college',
                                                                              'Assoc-voc',
                                                                              'Assoc-acdm',
                                                                              'Bachelors',
                                                                              'Masters',
                                                                              'Prof-school',
                                                                              'Doctorate']]),
                                                  ['education']),
                                                 ('Country-Encoded',
                                                  TargetEncoder(),
                                                  ['native-country'])])),
                ('Random Forest',
                 RandomForestClassifier(n_estimators=300, random_state=1))])

In [137]:
pipe.fit(X_train,y_train)

y_pred  = pipe.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)
print("Accuracy :",accuracy)

Accuracy : 0.8443034087419388


# **Creating Another Pipeline from Categorical and Numerical Separatly**

---



In [138]:
categorical_pipeline = Pipeline([
    ('encoder',OneHotEncoder(sparse_output=False))
])
categorical_pipeline

Pipeline(steps=[('encoder', OneHotEncoder(sparse_output=False))])

In [139]:
numeric_pipeline = Pipeline([
    ('Scaler',StandardScaler())
])
numeric_pipeline

Pipeline(steps=[('Scaler', StandardScaler())])

In [140]:
education_levels = [
    'Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th',
    'HS-grad', 'Some-college', 'Assoc-voc', 'Assoc-acdm', 'Bachelors', 'Masters',
    'Prof-school', 'Doctorate'
]

In [141]:
preprocessor = ColumnTransformer(
    [
        ('Numeric',numeric_pipeline,['age','capital-gain', 'capital-loss', 'hours-per-week']),
        ('Workclass_OHE', categorical_pipeline, ['workclass']),
        ('Country_Encoded',TargetEncoder(),['native-country']),
        ('Education_Encoded',OrdinalEncoder(categories=[education_levels]),['education'])
    ],
    remainder='passthrough'
)
preprocessor


ColumnTransformer(remainder='passthrough',
                  transformers=[('Numeric',
                                 Pipeline(steps=[('Scaler', StandardScaler())]),
                                 ['age', 'capital-gain', 'capital-loss',
                                  'hours-per-week']),
                                ('Workclass_OHE',
                                 Pipeline(steps=[('encoder',
                                                  OneHotEncoder(sparse_output=False))]),
                                 ['workclass']),
                                ('Country_Encoded', TargetEncoder(),
                                 ['native-country']),
                                ('Education_Encoded',
                                 OrdinalEncoder(categories=[['Preschool',
                                                             '1st-4th',
                                                             '5th-6th',
                                                             '7th-8th', '9th',
                                                             '10th', '11th',
                                                             '12th', 'HS-grad',
                                                             'Some-college',
                                                             'Assoc-voc',
                                                             'Assoc-acdm',
                                                             'Bachelors',
                                                             'Masters',
                                                             'Prof-school',
                                                             'Doctorate']]),
                                 ['education'])])

In [142]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Income Convert Binary
df['income_binary'] = df['income'].apply(lambda x: 1 if str(x).strip().lower() == '>50k' else 0)

# 2. Select Feature
X = df[['age','capital-gain', 'capital-loss', 'hours-per-week','workclass', 'education','native-country']]
y = df['income_binary']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=1)


model = make_pipeline(preprocessor,RandomForestClassifier(n_estimators=250,random_state=1))
model.fit(X_train,y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)

print("Accuracy :",accuracy)

Accuracy : 0.8190639970519592
